## 1. Setup — Add project root to Python path

In [ ]:
import sys, os

# Ensure the project root is on the path so `src.*` imports resolve
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.getcwd().endswith("src") else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Python path (first 3): {sys.path[:3]}")

## 2. Initialize the RAGGuard Pipeline

In [4]:
from src.core import RAGGuardPipeline

pipeline = RAGGuardPipeline()

Initializing RAGGuard Pipeline...
Pipeline ready! Using Apple MPS / CPU.



## 3. Define Input Texts

- **Source document** — the ground-truth context given to the LLM.
- **LLM-generated response** — contains subtle hallucinations (e.g. `200%` instead of `20%`, and a fabricated person `John Smith`).

In [5]:
source_document = """
Acme Corp released its Q3 earnings report yesterday. The company revenue grew by 20% compared to last year. 
The current CEO, Jane Doe, stated that the growth was primarily driven by their new AI software division. 
However, the hardware division saw a 5% decline in sales.
"""

# Subtle hallucinations:
#   - "200%" (should be 20%)
#   - "led by John Smith" (fabricated — not in source)
llm_generated_response = """
Acme Corp's revenue grew by 200% in Q3. The CEO, Jane Doe, attributed this to the AI software division. 
The hardware division was led by John Smith.
"""

print("Source document:")
print(source_document)
print("LLM-generated response:")
print(llm_generated_response)

Source document:

Acme Corp released its Q3 earnings report yesterday. The company revenue grew by 20% compared to last year. 
The current CEO, Jane Doe, stated that the growth was primarily driven by their new AI software division. 
However, the hardware division saw a 5% decline in sales.

LLM-generated response:

Acme Corp's revenue grew by 200% in Q3. The CEO, Jane Doe, attributed this to the AI software division. 
The hardware division was led by John Smith.



## 4. Run the Pipeline

In [6]:
report = pipeline.evaluate(
    generated_text=llm_generated_response,
    source_text=source_document
)

print(f"\nEvaluated {len(report)} claim(s).")


Evaluated 3 claim(s).


## 5. Visualise Results

In [7]:
import json
import pandas as pd
from IPython.display import display, HTML

# ── Colour-coded label badge ──────────────────────────────────────────────────
LABEL_COLOURS = {
    "Entailment":    "#28a745",  # green
    "Contradiction": "#dc3545",  # red
    "Neutral":       "#fd7e14",  # orange
}

def badge(label):
    colour = LABEL_COLOURS.get(label, "#6c757d")
    return f'<span style="background:{colour};color:#fff;padding:2px 8px;border-radius:4px;font-weight:bold">{label}</span>'

rows = []
for i, r in enumerate(report, 1):
    rows.append({
        "#":               i,
        "Claim":           r["claim"],
        "Matched Context": r["matched_context"],
        "NLI Label":       badge(r["nli_label"]),
        "Confidence":      f"{r['confidence']:.4f}",
    })

df = pd.DataFrame(rows)
display(HTML(df.to_html(escape=False, index=False)))

#,Claim,Matched Context,NLI Label,Confidence
1,Acme Corp's revenue grew by 200% in Q3.,Acme Corp released its Q3 earnings report yesterday. The company revenue grew by 20% compared to last year.,Contradiction,0.9974
2,"The CEO, Jane Doe, attributed this to the AI software division.","The current CEO, Jane Doe, stated that the growth was primarily driven by their new AI software division. However, the hardware division saw a 5% decline in sales.",Neutral,0.9847
3,The hardware division was led by John Smith.,"However, the hardware division saw a 5% decline in sales. The current CEO, Jane Doe, stated that the growth was primarily driven by their new AI software division.",Contradiction,0.9994


## 6. Summary Statistics

In [8]:
from collections import Counter

label_counts = Counter(r["nli_label"] for r in report)
avg_confidence = sum(r["confidence"] for r in report) / len(report) if report else 0

print("=" * 45)
print("  RAGGuard Evaluation Summary")
print("=" * 45)
print(f"  Total claims evaluated : {len(report)}")
for label in ["Entailment", "Contradiction", "Neutral"]:
    count = label_counts.get(label, 0)
    pct   = 100 * count / len(report) if report else 0
    print(f"  {label:<16}: {count}  ({pct:.0f}%)")
print(f"  Avg confidence     : {avg_confidence:.4f}")
print("=" * 45)

  RAGGuard Evaluation Summary
  Total claims evaluated : 3
  Entailment      : 0  (0%)
  Contradiction   : 2  (67%)
  Neutral         : 1  (33%)
  Avg confidence     : 0.9938


## 7. Save Results to JSON

In [ ]:
output = {
    "source_document":        source_document.strip(),
    "llm_generated_response": llm_generated_response.strip(),
    "summary": {
        "total_claims":    len(report),
        "label_counts":    dict(label_counts),
        "avg_confidence":  round(avg_confidence, 4),
    },
    "claims": report,
}

out_path = os.path.join(project_root, "rag_guard_results.json")
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to: {out_path}")